In [13]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from scipy.sparse import load_npz

In [2]:
FILES = {
    "tfidf_cleaned": ("text_representation/TF_IDF/text_representation_cleaned_tfidf.npz", "schemas/fully_cleaned.csv"),
    "tfidf_no_lemma": ("text_representation/TF_IDF/text_representation_tfidf_without_Lemmatization.npz", "schemas/without_lemmatization.csv"),
    "tfidf_no_stop": ("text_representation/TF_IDF/text_representation_tfidf_withoutStopwords_RemovePunctuation.npz", "schemas/without_stopwords_lowercasing_punct.csv"),

    "glove_cleaned": ("text_representation/Glove/text_representation_glove_cleaned.csv", "schemas/fully_cleaned.csv"),
    "glove_no_lemma": ("text_representation/Glove/text_representation_glove_without_Lemmatization.csv", "schemas/without_lemmatization.csv"),
    "glove_no_stop": ("text_representation/Glove/text_representation_glove_withoutStopwords_removePunctuation.csv", "schemas/without_stopwords_lowercasing_punct.csv")
}

## SVM

In [ ]:
for name, (feat_path, label_path) in FILES.items():

    df_labels = pd.read_csv(label_path)

    if feat_path.endswith(".npz"):
        X = load_npz(feat_path)
        y = df_labels["sentiment"]

    else:
        df_feat = pd.read_csv(feat_path)
        df_feat["sentiment"] = df_labels["sentiment"]

        X = df_feat.drop("sentiment", axis=1)
        y = df_feat["sentiment"]

    # # OPTIONAL: alignment safety
    # min_len = min(X.shape[0], len(y))
    # X = X[:min_len]
    # y = y[:min_len]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # ---- class weights FIX ----
    classes = np.unique(y)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )
    class_weights_dict = dict(zip(classes, weights))

    svm = SVC(class_weight=class_weights_dict, random_state=42)
    svm.fit(X_train, y_train)
    svm_pred = svm.predict(X_test)

    # ---- saving FIX ----
    output_df = pd.DataFrame({
        "sentiment": y_test,
        "svm_pred": svm_pred
    })

    output_df.to_csv(f"output/svm_predictions_{name}.csv", index=False)


In [15]:
df = pd.read_csv('output/svm_predictions_tfidf_cleaned.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("TF-IDF Cleaned SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF Cleaned SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF Cleaned SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF Cleaned SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.67      0.45      0.53       148
     neutral       0.59      0.64      0.61       316
    positive       0.85      0.88      0.86       923

    accuracy                           0.78      1387
   macro avg       0.70      0.65      0.67      1387
weighted avg       0.77      0.78      0.77      1387

[[ 66  40  42]
 [ 18 201  97]
 [ 15 100 808]]
TF-IDF Cleaned SVM F1 Score: 0.7715030007977556
TF-IDF Cleaned SVM Precision: 0.7732179520972821
TF-IDF Cleaned SVM Recall: 0.7750540735400144
TF-IDF Cleaned SVM Accuracy Score: 0.7750540735400144


In [16]:
df = pd.read_csv('output/svm_predictions_tfidf_no_lemma.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("TF-IDF No Lemma SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Lemma SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Lemma SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Lemma SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.71      0.46      0.56       148
     neutral       0.59      0.66      0.62       316
    positive       0.86      0.88      0.87       923

    accuracy                           0.78      1387
   macro avg       0.72      0.67      0.68      1387
weighted avg       0.79      0.78      0.78      1387

[[ 68  42  38]
 [ 18 208  90]
 [ 10 101 812]]
TF-IDF No Lemma SVM F1 Score: 0.7816638111521518
TF-IDF No Lemma SVM Precision: 0.7854415906341846
TF-IDF No Lemma SVM Recall: 0.7844268204758471
TF-IDF No Lemma SVM Accuracy Score: 0.7844268204758471


In [17]:
df = pd.read_csv('output/svm_predictions_tfidf_no_stop.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("TF-IDF No Stopwords SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Stopwords SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Stopwords SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("TF-IDF No Stopwords SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.74      0.49      0.59       155
     neutral       0.58      0.62      0.60       312
    positive       0.85      0.88      0.87       920

    accuracy                           0.78      1387
   macro avg       0.72      0.66      0.68      1387
weighted avg       0.78      0.78      0.77      1387

[[ 76  38  41]
 [ 19 193 100]
 [  8 103 809]]
TF-IDF No Stopwords SVM F1 Score: 0.7741647183949093
TF-IDF No Stopwords SVM Precision: 0.7772955939203193
TF-IDF No Stopwords SVM Recall: 0.7772170151405912
TF-IDF No Stopwords SVM Accuracy Score: 0.7772170151405912


In [18]:
df = pd.read_csv('output/svm_predictions_glove_cleaned.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("Glove Cleaned SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove Cleaned SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove Cleaned SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove Cleaned SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.33      0.58      0.42       148
     neutral       0.34      0.56      0.42       316
    positive       0.85      0.57      0.68       923

    accuracy                           0.57      1387
   macro avg       0.51      0.57      0.51      1387
weighted avg       0.68      0.57      0.59      1387

[[ 86  40  22]
 [ 69 176  71]
 [102 297 524]]
Glove Cleaned SVM F1 Score: 0.594917027620264
Glove Cleaned SVM Precision: 0.6790305483259937
Glove Cleaned SVM Recall: 0.5666906993511175
Glove Cleaned SVM Accuracy Score: 0.5666906993511175


In [19]:
df = pd.read_csv('output/svm_predictions_glove_no_lemma.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("Glove No Lemma SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Lemma SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Lemma SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Lemma SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.36      0.61      0.46       148
     neutral       0.35      0.57      0.44       316
    positive       0.85      0.57      0.68       923

    accuracy                           0.58      1387
   macro avg       0.52      0.59      0.52      1387
weighted avg       0.68      0.58      0.60      1387

[[ 91  35  22]
 [ 63 180  73]
 [ 98 296 529]]
Glove No Lemma SVM F1 Score: 0.6028414808915104
Glove No Lemma SVM Precision: 0.6829378267555422
Glove No Lemma SVM Recall: 0.5767844268204758
Glove No Lemma SVM Accuracy Score: 0.5767844268204758


In [20]:
df = pd.read_csv('output/svm_predictions_glove_no_stop.csv')
print(classification_report(df['sentiment'], df['svm_pred']))
print(confusion_matrix(df['sentiment'], df['svm_pred']))
print("Glove No Stopwords SVM F1 Score:", f1_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Stopwords SVM Precision:", precision_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Stopwords SVM Recall:", recall_score(df['sentiment'], df['svm_pred'], average='weighted'))
print("Glove No Stopwords SVM Accuracy Score:", accuracy_score(df['sentiment'], df['svm_pred']))

              precision    recall  f1-score   support

    negative       0.31      0.60      0.41       155
     neutral       0.33      0.57      0.42       312
    positive       0.90      0.53      0.67       920

    accuracy                           0.55      1387
   macro avg       0.51      0.57      0.50      1387
weighted avg       0.70      0.55      0.58      1387

[[ 93  49  13]
 [ 89 179  44]
 [119 313 488]]
Glove No Stopwords SVM F1 Score: 0.5818913196324039
Glove No Stopwords SVM Precision: 0.7028848129536429
Glove No Stopwords SVM Recall: 0.547945205479452
Glove No Stopwords SVM Accuracy Score: 0.547945205479452


## Linear Regression

In [ ]:
for name, (feat_path, label_path) in FILES.items():

    df_labels = pd.read_csv(label_path)

    if feat_path.endswith(".npz"):
        X = load_npz(feat_path)
        y = df_labels["sentiment"]

    else:
        df_feat = pd.read_csv(feat_path)
        df_feat["sentiment"] = df_labels["sentiment"]

        X = df_feat.drop("sentiment", axis=1)
        y = df_feat["sentiment"]

    mapping = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
    }

    y = y.map(mapping)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # ---- class weights FIX ----
    classes = np.unique(y)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )
    class_weights_dict = dict(zip(classes, weights))
    sample_weights = y.map(class_weights_dict)
    
    
    ln = LinearRegression()
    ln.fit(X_train, y_train, sample_weight=sample_weights.loc[y_train.index])
    ln_pred = ln.predict(X_test)
    
    def to_class(x):
        if x < -0.5:
            return -1
        elif x > 0.5:
            return 1
        else:
            return 0
    
    y_test = y_test.reset_index(drop=True)
    ln_pred_class = np.vectorize(to_class)(ln_pred)
    ln_pred_class = pd.Series(ln_pred_class).reset_index(drop=True)
    # reverse_mapping
    ln_pred_class = ln_pred_class.map({-1: "negative", 0: "neutral", 1: "positive"})
    y_test_class = y_test.map({-1: "negative", 0: "neutral", 1: "positive"})
    # ---- saving FIX ----
    output_df = pd.DataFrame({
        "sentiment": y_test_class,
        "ln_pred_class": ln_pred_class
    })

    output_df.to_csv(f"output/ln_predictions_{name}.csv", index=False)



In [36]:
df = pd.read_csv('output/ln_predictions_tfidf_cleaned.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("TF-IDF Cleaned Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF Cleaned Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF Cleaned Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF Cleaned Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.13      0.54      0.21       186
     neutral       0.54      0.09      0.16       394
    positive       0.69      0.54      0.61      1153

    accuracy                           0.44      1733
   macro avg       0.45      0.39      0.33      1733
weighted avg       0.60      0.44      0.46      1733

[[101   6  79]
 [159  36 199]
 [500  25 628]]
TF-IDF Cleaned Linear Regression F1 Score: 0.4642747221382166
TF-IDF Cleaned Linear Regression Precision: 0.5975935525760113
TF-IDF Cleaned Linear Regression Recall: 0.4414310444316214
TF-IDF Cleaned Linear Regression Accuracy Score: 0.44143104443162146


In [32]:
df = pd.read_csv('output/ln_predictions_tfidf_no_lemma.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("TF-IDF No Lemma Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Lemma Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Lemma Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Lemma Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.14      0.61      0.23       186
     neutral       0.66      0.09      0.16       394
    positive       0.72      0.54      0.62      1153

    accuracy                           0.45      1733
   macro avg       0.51      0.41      0.34      1733
weighted avg       0.64      0.45      0.47      1733

[[113   1  72]
 [184  37 173]
 [510  18 625]]
TF-IDF No Lemma Linear Regression F1 Score: 0.47291140740408705
TF-IDF No Lemma Linear Regression Precision: 0.6432029059076777
TF-IDF No Lemma Linear Regression Recall: 0.447201384881708
TF-IDF No Lemma Linear Regression Accuracy Score: 0.447201384881708


In [33]:
df = pd.read_csv('output/ln_predictions_tfidf_no_stop.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("TF-IDF No Stopwords Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Stopwords Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Stopwords Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("TF-IDF No Stopwords Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.11      0.44      0.18       198
     neutral       0.42      0.08      0.13       391
    positive       0.69      0.53      0.60      1145

    accuracy                           0.42      1734
   macro avg       0.41      0.35      0.30      1734
weighted avg       0.56      0.42      0.45      1734

[[ 87   7 104]
 [191  30 170]
 [504  34 607]]
TF-IDF No Stopwords Linear Regression F1 Score: 0.44523072549173176
TF-IDF No Stopwords Linear Regression Precision: 0.5629369686635872
TF-IDF No Stopwords Linear Regression Recall: 0.41753171856978083
TF-IDF No Stopwords Linear Regression Accuracy Score: 0.41753171856978083


In [34]:
df = pd.read_csv('output/ln_predictions_glove_cleaned.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("Glove Cleaned Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove Cleaned Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove Cleaned Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove Cleaned Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.51      0.22      0.31       186
     neutral       0.26      0.89      0.40       394
    positive       0.91      0.24      0.38      1153

    accuracy                           0.38      1733
   macro avg       0.56      0.45      0.36      1733
weighted avg       0.72      0.38      0.37      1733

[[ 41 143   2]
 [ 20 349  25]
 [ 19 861 273]]
Glove Cleaned Linear Regression F1 Score: 0.37393285876281174
Glove Cleaned Linear Regression Precision: 0.7190914315348892
Glove Cleaned Linear Regression Recall: 0.3825735718407386
Glove Cleaned Linear Regression Accuracy Score: 0.3825735718407386


In [ ]:
df = pd.read_csv('output/ln_predictions_glove_no_lemma.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("Glove No Lemma Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Lemma Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Lemma Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Lemma Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         2
     neutral       0.50      0.33      0.40         3
    positive       0.50      0.80      0.62         5

    accuracy                           0.50        10
   macro avg       0.33      0.38      0.34        10
weighted avg       0.40      0.50      0.43        10

[[0 0 2]
 [0 1 2]
 [0 1 4]]
Glove No Lemma Linear Regression F1 Score: 0.42769230769230776
Glove No Lemma Linear Regression Precision: 0.4
Glove No Lemma Linear Regression Recall: 0.5
Glove No Lemma Linear Regression Accuracy Score: 0.5


/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/muhammed_mahmoud/Projects/python_env/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [ ]:
df = pd.read_csv('output/ln_predictions_glove_no_stop.csv')
print(classification_report(df['sentiment'], df['ln_pred_class']))
print(confusion_matrix(df['sentiment'], df['ln_pred_class']))
print("Glove No Stopwords Linear Regression F1 Score:", f1_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Stopwords Linear Regression Precision:", precision_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Stopwords Linear Regression Recall:", recall_score(df['sentiment'], df['ln_pred_class'], average='weighted'))
print("Glove No Stopwords Linear Regression Accuracy Score:", accuracy_score(df['sentiment'], df['ln_pred_class']))

              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         2
     neutral       0.50      0.67      0.57         3
    positive       0.60      0.60      0.60         5

    accuracy                           0.50        10
   macro avg       0.37      0.42      0.39        10
weighted avg       0.45      0.50      0.47        10

[[0 1 1]
 [0 2 1]
 [1 1 3]]
Glove No Stopwords Linear Regression F1 Score: 0.4714285714285714
Glove No Stopwords Linear Regression Precision: 0.45
Glove No Stopwords Linear Regression Recall: 0.5
Glove No Stopwords Linear Regression Accuracy Score: 0.5
